In [2]:
# =====================================================
# STEP 0: Mount Google Drive
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this path to where you have your video in Drive
os.chdir("/content/drive/MyDrive/")  # adjust according to your folder

# =====================================================
# STEP 1: Extract frames from video
# =====================================================
import cv2
import numpy as np

VIDEO_FILE = 'video.mp4'   # <-- put your file here
FPS_OBJETIVO = 24
SEGUNDOS = 10
TOTAL_FRAMES = FPS_OBJETIVO * SEGUNDOS   # = 240
ANCHO = 320
ALTO  = 240

vidcap = cv2.VideoCapture(VIDEO_FILE)
fps_original = vidcap.get(cv2.CAP_PROP_FPS)
total_frames_video = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Original video FPS: {fps_original}")
print(f"Total frames in video: {total_frames_video}")
print(f"Duration: {total_frames_video/fps_original:.2f} seconds")

# We calculate which frames to take to get exactly 24 FPS for 10s
# even if the original video has a different FPS
indices_a_tomar = np.linspace(0, min(total_frames_video - 1,
                               int(fps_original * SEGUNDOS) - 1),
                               TOTAL_FRAMES, dtype=int)

frames_rgb565 = []

for i, frame_idx in enumerate(indices_a_tomar):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    success, frame = vidcap.read()
    if not success:
        print(f"Error reading frame {frame_idx}, using black frame")
        frame = np.zeros((ALTO, ANCHO, 3), dtype=np.uint8)

    # Resize to 320x240
    frame_resized = cv2.resize(frame, (ANCHO, ALTO))

    # OpenCV uses BGR, convert to RGB
    frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

    # Convert to RGB565 (16 bits)
    # R: bits 15-11 (5 bits), G: bits 10-5 (6 bits), B: bits 4-0 (5 bits)
    r = (frame_rgb[:, :, 0].astype(np.uint16) >> 3) & 0x1F
    g = (frame_rgb[:, :, 1].astype(np.uint16) >> 2) & 0x3F
    b = (frame_rgb[:, :, 2].astype(np.uint16) >> 3) & 0x1F
    rgb565 = (r << 11) | (g << 5) | b   # each value is uint16

    frames_rgb565.append(rgb565)

    if i % 24 == 0:
        print(f"  Processed frame {i+1}/{TOTAL_FRAMES}")

vidcap.release()
print(f"\nTotal frames processed: {len(frames_rgb565)}")
print(f"Estimated total memory: {len(frames_rgb565) * ANCHO * ALTO * 2 / 1024 / 1024:.1f} MB")

# =====================================================
# STEP 2: Generate the Assembly file
# IMPORTANT: writing BACKWARDS
# Last frame first, first frame last
# This way, when reading .data sequentially, it plays in correct order
# =====================================================
print("\nGenerating video_data.s file ...")

OUTPUT_FILE = "video_data.s"

with open(OUTPUT_FILE, "w") as f:
    f.write("# ============================================================\n")
    f.write("# Video data: 240 frames, 320x240, RGB565\n")
    f.write("# Automatically generated — DO NOT EDIT\n")
    f.write(f"# Total frames: {TOTAL_FRAMES}, Resolution: {ANCHO}x{ALTO}\n")
    f.write(f"# Total bytes: {TOTAL_FRAMES * ANCHO * ALTO * 2}\n")
    f.write("# ============================================================\n\n")
    f.write(".section .rodata\n")
    f.write(".align 2\n")
    f.write(".global video_data_start\n")
    f.write(".global video_data_end\n\n")
    f.write("video_data_start:\n")

    # Write frames BACKWARDS
    # frame[239] first, frame[0] last
    for i in range(TOTAL_FRAMES - 1, -1, -1):
        frame = frames_rgb565[i]
        f.write(f"  # --- Frame {i} ---\n")
        # Each row of the frame
        # We also write rows backwards within the frame
        for row in range(ALTO - 1, -1, -1):
            pixels = frame[row, :]
            # Pack 2 pixels per .word (32 bits)
            # even pixel in bits 31-16, odd pixel in bits 15-0
            for col in range(0, ANCHO, 2):
                p0 = int(pixels[col])
                p1 = int(pixels[col + 1]) if col + 1 < ANCHO else 0
                word_val = (p0 << 16) | p1
                f.write(f"  .word 0x{word_val:08X}\n")

    f.write("\nvideo_data_end:\n")
    f.write("  .word 0x00000000\n")

print(f"File generated: {OUTPUT_FILE}")
print(f"Approximate .s size: {os.path.getsize(OUTPUT_FILE)/1024/1024:.1f} MB")
print("\nDone! Copy video_data.s to your project in VS Code.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original video FPS: 29.426751592356688
Total frames in video: 308
Duration: 10.47 seconds
  Processed frame 1/240
  Processed frame 25/240
  Processed frame 49/240
  Processed frame 73/240
  Processed frame 97/240
  Processed frame 121/240
  Processed frame 145/240
  Processed frame 169/240
  Processed frame 193/240
  Processed frame 217/240

Total frames processed: 240
Estimated total memory: 35.2 MB

Generating video_data.s file ...
File generated: video_data.s
Approximate .s size: 167.0 MB

Done! Copy video_data.s to your project in VS Code.
